# Test dei 4 modelli addestrati (NSynth, riconoscimento della nota)

Carica i **pesi addestrati** (`pesi_nota.npz`: 4 modelli — Dynamic GD,
Newton-CG, Newton-CG L1, BB-CCV — regressione logistica multinomiale a 12
classi di altezza, addestrati su NSynth `valid` con `MAX_ITER=300`, seed 42,
come nella Sezione 6.5 della tesi) insieme alla **standardizzazione** delle
features (media e deviazione standard del training, salvate nello stesso
file) e verifica le previsioni su una **clip audio**.

Puoi testare in due modi:
- su un **qualsiasi file `.wav`** (basta caricarlo: cella 4),
- su una **clip del test set NSynth** (caricando `features_nota.npz`, cella 6).

**File da caricare** (dalla tua cartella locale):
- `pesi_nota.npz` (obbligatorio: pesi dei 4 modelli + `mu`/`sd`)
- `features_nota.npz` (facoltativo: accuratezza sul test set e clip singole)

> Come ottenere i file: esegui il notebook `nsynth_nota_riproduzione`
> (Sezione 6.5) fino in fondo: la cella "4b. Salva i pesi..." li scarica
> automaticamente.

In [ ]:
#@title 0. Dipendenze e import
%pip install -q librosa
import numpy as np
import librosa
import json, os
print("librosa", librosa.__version__, "| numpy", np.__version__)


In [ ]:
#@title 1. Carica i pesi addestrati (upload da locale)
from google.colab import files
print("Carica pesi_nota.npz (pesi dei 4 modelli + mu/sd):")
f1 = files.upload()

PESI = np.load(list(f1.keys())[0])
MU, SD = PESI["mu"], PESI["sd"]
C = int(PESI["C"])
NOTE = [str(s) for s in PESI["NOTE"]]

DISPLAY = {"Dynamic_GD": "Dynamic GD", "Newton-CG": "Newton-CG",
           "Newton-CG_L1": "Newton-CG L1", "BB-CCV": "BB-CCV"}
print("\nModelli caricati:", list(DISPLAY.keys()))
print("Pesi shape:", {k: PESI[k].shape for k in DISPLAY.keys()})
print("Note (12 classi):", NOTE)


In [ ]:
#@title 2. Modello (regressione logistica multinomiale, come nella Sez. 6.5)
def _W(w):
    return w.reshape(C, -1)          # W in R^{12 x 25}: 24 features + bias

def softmax(Z):
    Z = Z - Z.max(axis=-1, keepdims=True)
    P = np.exp(Z)
    return P / P.sum(axis=-1, keepdims=True)

def prob_pred(w, x):
    """Probabilita' sulle 12 note per una clip x (gia' standardizzata, 24 dim)."""
    x_aug = np.concatenate([np.asarray(x, dtype=float), [1.0]])  # bias
    return softmax(_W(w) @ x_aug)


In [ ]:
#@title 3. Features chroma 24-dim (identiche al notebook di riproduzione)
SR = 16000

def estrai_chroma(wav_path):
    y, _ = librosa.load(wav_path, sr=SR, mono=True)
    Cm = librosa.feature.chroma_stft(y=y, sr=SR)              # 12 bin cromatici
    return np.concatenate([Cm.mean(axis=1), Cm.std(axis=1)])  # 24 dim

def standardizza(x):
    return (np.asarray(x, dtype=float) - MU) / SD


In [ ]:
#@title 4. Test su una clip audio (.wav, qualsiasi file)
from google.colab import files
print("Carica una clip .wav di una nota (singola, monofonica, ~4 s):")
f2 = files.upload()
wav = list(f2.keys())[0]

x = standardizza(estrai_chroma(wav))
print(f"\nClip: {wav}  |  features: {x.shape[0]} dim\n")
print(f"{'Modello':14s} {'Nota pred.':10s}  {'P (pred)':>8s}   top-3")
print("-" * 56)
for key, disp in DISPLAY.items():
    P = prob_pred(PESI[key], x)
    pred = int(np.argmax(P))
    top3 = sorted(zip(NOTE, P), key=lambda t: -t[1])[:3]
    s3 = ", ".join(f"{n}:{p:.2f}" for n, p in top3)
    print(f"{disp:14s} {NOTE[pred]:10s}  {P[pred]:8.1%}   {s3}")
print()
print("Note possibili:", ", ".join(NOTE))
print("(Il modello e' addestrato su note singole monofoniche: per clip")
print(" polifoniche o molto lunghe la predizione e' meno affidabile.)")


In [ ]:
#@title 5. (Facoltativo) Ascolta la clip caricata
from IPython.display import Audio, display
display(Audio(wav))


In [ ]:
#@title 6. (Facoltativo) Accuratezza sul test set NSynth
# Carica features_nota.npz (split test) -> accuratezza dei 4 modelli.
from google.colab import files
print("Carica features_nota.npz (Xte gia' standardizzato, Yte, names):")
f3 = files.upload()
z = np.load(list(f3.keys())[0])
Xte, Yte = z["Xte"], z["Yte"]
Xa = np.hstack([Xte, np.ones((Xte.shape[0], 1))])
print(f"Test: {Xte.shape[0]} clip\n")
print(f"{'Modello':14s} {'Acc. test':>9s}")
print("-" * 28)
for key, disp in DISPLAY.items():
    W = _W(PESI[key])
    acc = float(np.mean(np.argmax(Xa @ W.T, axis=1) == Yte))
    print(f"{disp:14s} {acc*100:8.1f}%")


In [ ]:
#@title 7. Predizioni su singole clip del test set (nota vera)
# Usa features_nota.npz (gia' caricato nella cella 6, oppure caricalo qui).
if 'z' not in globals():
    from google.colab import files
    print("Carica features_nota.npz:")
    fz = files.upload()
    z = np.load(list(fz.keys())[0])
Xte, Yte, names = z["Xte"], z["Yte"], z["names"]

sel = input(f"Inserisci indice clip (0..{len(Xte)-1}) o parte del nome: ").strip()
if sel.isdigit():
    i = int(sel)
else:
    match = [sel.lower() in n.lower() for n in names]
    if not any(match):
        print(f"Nessuna clip contiene '{sel}' (esempio: {names[0]}). Uso l'indice 0.")
        i = 0
    else:
        i = int(np.argmax(match))
print("\nClip:", names[i], "| nota VERA:", NOTE[Yte[i]], "\n")
Xa = np.hstack([Xte[[i]], np.ones((1, 1))])
for key, disp in DISPLAY.items():
    W = _W(PESI[key])
    P = softmax(Xa @ W.T)[0]
    pred = int(np.argmax(P))
    ok = "OK" if NOTE[pred] == NOTE[Yte[i]] else "NO"
    top3 = sorted(zip(NOTE, P), key=lambda t: -t[1])[:3]
    s3 = ", ".join(f"{n}:{p:.2f}" for n, p in top3)
    print(f"{disp:14s} {NOTE[pred]:10s} {P[pred]:6.1%}  [{ok}]  top3: {s3}")


In [ ]:
#@title 8. Ascolta la clip del test set (scarica l'audio NSynth)
# Scarica nsynth-test.jsonwav.tar.gz (~350 MB) UNA volta per sessione ed
# estrae SOLO la clip selezionata nella cella 7, poi la riproduce.
import tarfile, urllib.request
from IPython.display import Audio, display

nome = names[i] if ('i' in globals() and i < len(names)) else \
       "keyboard_electronic_078-053-050"
wav = f"nsynth-test/audio/{nome}.wav"
TARGZ = "nsynth-test.jsonwav.tar.gz"
URL = "http://download.magenta.tensorflow.org/datasets/nsynth/" + TARGZ

if not os.path.exists(wav):
    if not os.path.exists(TARGZ):
        print(f"Scarico {TARGZ} (~350 MB, una volta per sessione)...")
        urllib.request.urlretrieve(URL, TARGZ)
    print(f"Estraggo {wav} dall'archivio...")
    with tarfile.open(TARGZ, "r:gz") as tar:
        for m in tar.getmembers():
            if m.name == wav:
                tar.extract(m, ".")
                break
    if not os.path.exists(wav):
        raise FileNotFoundError(f"{wav} non trovato nell'archivio (nome errato?)")

print(f"Riproduzione: {nome}")
display(Audio(wav))
